# 05 - Pagination, Filtering and Sorting

## 1. Why Pagination?

**Definition:** Pagination means returning data in **small pages** instead of all rows at once.

**Why:** Returning 1 million orders in one response is slow, uses huge memory, and crashes the client.

## 2. Offset Pagination

**How it works:** The client sends page and size. The server skips some rows and takes the next batch.

**Formula:** skip = (page - 1) × size

**Example:** page=3, size=10 → skip 20 rows, return rows 21–30
SQL: SELECT * FROM orders ORDER BY id LIMIT 10 OFFSET 20

**Pros:** Simple, can jump to any page.
**Cons:** Slow on huge tables (database still scans skipped rows). If new rows are added, items can repeat or be skipped between pages.

## 3. Cursor Pagination

**How it works:** The server returns a **cursor** (usually the last id seen). The client sends it back to get the next page.

**Example:** GET /orders?after_id=120&size=10
SQL: SELECT * FROM orders WHERE id > 120 ORDER BY id LIMIT 10

**Pros:** Fast on huge tables, no duplicates when data changes.
**Cons:** Can't jump to page 50 directly.

**Real-world example:** An admin orders table uses offset (page numbers). An Instagram-style infinite feed uses cursor.

**Rule:** Always cap the page size (e.g. max 100) so no one requests size=1000000.

**Interview line:** "I use offset pagination for small admin tables, cursor pagination for large or fast-changing data like feeds, and I always cap the page size."

In [ ]:
# ==================================================
# 1. OFFSET vs CURSOR PAGINATION
# ==================================================
orders = [{"id": i, "status": "paid" if i % 2 else "pending"} for i in range(1, 51)]
MAX_SIZE = 100

def offset_page(page, size):
    size = min(size, MAX_SIZE)
    skip = (page - 1) * size
    return orders[skip:skip + size]

def cursor_page(after_id, size):
    size = min(size, MAX_SIZE)
    result = [o for o in orders if o["id"] > after_id][:size]
    next_cursor = result[-1]["id"] if result else None
    return result, next_cursor

print("=" * 55)
print("[OFFSET] page=3, size=10")
page = offset_page(3, 10)
print("  ids:", [o["id"] for o in page])

print("[CURSOR] after_id=0, size=10")
items, cursor = cursor_page(0, 10)
print("  ids:", [o["id"] for o in items], "| next cursor:", cursor)

print("[CURSOR] after_id=10, size=10")
items, cursor = cursor_page(cursor, 10)
print("  ids:", [o["id"] for o in items], "| next cursor:", cursor)

print("[CAP]    size=5000 requested ->", len(offset_page(1, 5000)), "rows returned (only 50 exist, capped at 100)")
print("=" * 55)

## 4. Filtering and Sorting

**Definition:** Filtering narrows the results, sorting orders them. Both go in **query parameters**.

**Common patterns:**
- Filter: /orders?status=paid&city=Hyderabad
- Range: /orders?min_amount=500&max_amount=2000
- Sort ascending: /orders?sort=date
- Sort descending: /orders?sort=-date (minus means descending)
- Combine with pagination: /orders?status=paid&sort=-date&page=1&size=20

**Rules:**
- Only allow sorting on known fields (whitelist), never pass user input straight into SQL
- Apply filter → sort → paginate, in that order

**Real-world example:** Amazon's "Price: Low to High" with a 4-star filter is filtering plus sorting in query parameters.

**Interview line:** "Filters and sorting go in query parameters, I whitelist sortable fields to avoid SQL injection, and I apply filter, then sort, then paginate."

In [ ]:
# ==================================================
# 2. FILTER -> SORT -> PAGINATE
# ==================================================
orders = [
    {"id": 1, "status": "paid",    "amount": 1200, "date": "2026-10-01"},
    {"id": 2, "status": "pending", "amount": 300,  "date": "2026-10-03"},
    {"id": 3, "status": "paid",    "amount": 800,  "date": "2026-10-05"},
    {"id": 4, "status": "paid",    "amount": 2500, "date": "2026-10-02"},
    {"id": 5, "status": "paid",    "amount": 450,  "date": "2026-10-04"},
]
ALLOWED_SORT = {"date", "amount"}

def list_orders(status=None, sort="date", page=1, size=2):
    field = sort.lstrip("-")
    if field not in ALLOWED_SORT:
        return f"[400] Cannot sort by '{field}'"
    result = [o for o in orders if status is None or o["status"] == status]   # filter
    result.sort(key=lambda o: o[field], reverse=sort.startswith("-"))         # sort
    skip = (page - 1) * size
    return result[skip:skip + size]                                           # paginate

print("=" * 55)
print("[REQUEST] ?status=paid&sort=-amount&page=1&size=2")
for o in list_orders(status="paid", sort="-amount"):
    print("  ", o)
print("[REQUEST] ?sort=password")
print("  ", list_orders(sort="password"))
print("=" * 55)

## Key Takeaways

- Never return all rows; paginate
- Offset: skip = (page - 1) × size, simple, slow on huge tables
- Cursor: WHERE id > last_id, fast, no duplicates, no page jumping
- Always cap page size
- Filters and sorting go in query parameters; "-" means descending
- Whitelist sortable fields; apply filter → sort → paginate